# W1 · Day 3 (v2) — Ensemble + TTA + Calibration (final within-one push)

Combines the three stabilized seeds per fold into an ensemble, applies test-time
augmentation, and fits a per-dataset decision-threshold calibration on the
validation split (applied once to test). No training occurs. Uses the w1v2_
checkpoints from Day 1 + Day 2. Leak guard runs before evaluation; calibration is
validation-only; all metrics reported together. Writes only to scope3_w1.

## Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import sys, importlib, os
sys.path.insert(0,'/content/drive/MyDrive/Master Thesis/scope3')
import config; importlib.reload(config)
import numpy as np, pandas as pd, json, glob
from pathlib import Path
import torch
if 'training_lib_max' in sys.modules: importlib.reload(sys.modules['training_lib_max'])
import training_lib_max as TM
device='cuda' if torch.cuda.is_available() else 'cpu'
if device!='cuda': raise RuntimeError('No GPU.')
PROJECT_ROOT=Path('/content/drive/MyDrive/Master Thesis')
W1_ROOT=PROJECT_ROOT/'scope3_w1'; W1_CKPT=W1_ROOT/'checkpoints'; W1_RES=W1_ROOT/'results'
manifest=TM.prepare_local_data()
def derive_patient_id(row):
    fn=str(row['filename']); stem=os.path.splitext(os.path.basename(fn))[0]
    for sep in ['_','-','.']:
        if sep in stem: stem=stem.split(sep)[0]; break
    return f"{row['dataset']}::{stem}"
manifest=manifest.copy(); manifest['patient_id']=manifest.apply(derive_patient_id,axis=1)
def within1(yt,yp): return float((np.abs(np.asarray(yt)-np.asarray(yp))<=1).mean())
print('Day-3 (v2) setup ready.')

Mounted at /content/drive
Copied array in 59s
Loaded array (61558, 224, 224) in 1s
Day-3 (v2) setup ready.


## Ensemble probabilities (TTA, averaged over seeds)

In [2]:
@torch.no_grad()
def ensemble_probs(test_ds, df, bs=12):
    cks=sorted(glob.glob(str(W1_CKPT/f'w1v2_{test_ds}_seed*_best.pt')))
    if not cks: cks=sorted(glob.glob(str(W1_CKPT/f'w1v2_{test_ds}_seed*.pt')))
    assert cks, f'No w1v2 checkpoints for {test_ds}'
    acc=None; n=0
    for ck in cks:
        m=TM.OrdinalNet(config.NUM_CLASSES,4,True).to(device)
        try: TM.load_ckpt(ck,m,None)
        except Exception as e: print(f'  skip {os.path.basename(ck)} ({type(e).__name__})'); continue
        _,p=TM.predict_tta(m,df,device,bs,use_tta=True); acc=p if acc is None else acc+p; n+=1
        del m; torch.cuda.empty_cache()
    return acc/max(1,n), n
print('Ensemble extractor ready.')

Ensemble extractor ready.


## Per-dataset threshold calibration (validation-only fit)

In [3]:
def decode_thr(cum, th): return (cum > np.asarray(th)[None,:]).sum(1)
def cum_from_cls(cls):
    c=np.cumsum(cls[:,::-1],axis=1)[:,::-1]; return c[:,1:]
def fit_thr(val_cum,y):
    th=[0.5,0.5,0.5,0.5]; grid=np.linspace(0.3,0.7,9)
    for _ in range(3):
        for k in range(4):
            bw=-1; bt=th[k]
            for t in grid:
                cand=th.copy(); cand[k]=t; w=within1(y,decode_thr(val_cum,cand))
                if w>bw: bw=w; bt=t
            th[k]=bt
    return th
print('Calibration ready.')

Calibration ready.


## Run per fold

In [4]:
from sklearn.metrics import accuracy_score, cohen_kappa_score
def finalize(test_ds):
    tr,va,te=TM.make_splits(manifest,test_ds,seed=0)
    assert test_ds not in set(tr['dataset'].unique())
    assert len(set(tr['patient_id'])&set(te['patient_id']))==0
    vcp,n=ensemble_probs(test_ds,va); tcp,_=ensemble_probs(test_ds,te)
    yv=va['kl_grade'].values; yt=te['kl_grade'].values
    bp=tcp.argmax(1); bw=within1(yt,bp)
    th=fit_thr(cum_from_cls(vcp),yv); cp=decode_thr(cum_from_cls(tcp),th); cw=within1(yt,cp)
    out={'test_ds':test_ds,'n_seeds':n,
         'ensemble_within1':bw,'ensemble_exact':float(accuracy_score(yt,bp)),'ensemble_qwk':float(cohen_kappa_score(yt,bp,weights='quadratic')),
         'calibrated_within1':cw,'calibrated_exact':float(accuracy_score(yt,cp)),'calibrated_qwk':float(cohen_kappa_score(yt,cp,weights='quadratic')),
         'thresholds':[round(float(x),3) for x in th],'grades_used':int(len(np.unique(cp)))}
    json.dump(out,open(str(W1_RES/f'w1v2_{test_ds}_final.json'),'w'),indent=2)
    print(f"[{test_ds}] seeds={n}")
    print(f"   ensemble    : within1={bw:.3f} exact={out['ensemble_exact']:.3f} qwk={out['ensemble_qwk']:.3f}")
    print(f"   +calibrated : within1={cw:.3f} exact={out['calibrated_exact']:.3f} qwk={out['calibrated_qwk']:.3f} thr={out['thresholds']} grades={out['grades_used']}/5")
    return out
final={}
for ds in ['oai','nhanes3']:
    print(f'=== {ds} ==='); final[ds]=finalize(ds)

=== oai ===
Downloading: "https://download.pytorch.org/models/convnext_large-ea097f82.pth" to /root/.cache/torch/hub/checkpoints/convnext_large-ea097f82.pth


100%|██████████| 755M/755M [00:17<00:00, 44.4MB/s]


[oai] seeds=3
   ensemble    : within1=0.830 exact=0.539 qwk=0.613
   +calibrated : within1=0.947 exact=0.303 qwk=0.518 thr=[0.3, 0.65, 0.5, 0.55] grades=5/5
=== nhanes3 ===
[nhanes3] seeds=3
   ensemble    : within1=0.759 exact=0.536 qwk=0.508
   +calibrated : within1=0.904 exact=0.242 qwk=0.515 thr=[0.3, 0.7, 0.45, 0.7] grades=4/5


## Final scorecard — the definitive within-one number

In [5]:
def seed0(ds):
    p=W1_RES/f'w1v2_{ds}_seed0.json'; return json.load(open(str(p)))['external_within1'] if p.exists() else float('nan')
print('OPTION C FINAL — within-one (single seed -> ensemble -> +calibration):')
print(f"{'fold':10s}{'seed0':>10s}{'ensemble':>11s}{'+calib':>10s}{'exact':>9s}{'qwk':>8s}")
best=0
for ds in ['oai','nhanes3']:
    f=final[ds]; print(f"{ds:10s}{seed0(ds):>10.3f}{f['ensemble_within1']:>11.3f}{f['calibrated_within1']:>10.3f}{f['calibrated_exact']:>9.3f}{f['calibrated_qwk']:>8.3f}")
    best=max(best,f['calibrated_within1'])
print()
print(f'Best within-1 reached: {best:.3f}')
if best>=0.95: print('>>> 0.95 REACHED.')
elif best>=0.88: print('>>> 0.88-0.94 band: strong, but short of 0.95.')
else: print(f'>>> Ceiling confirmed ~{best:.2f}. Even 3-seed ensemble + TTA + calibration does not reach 0.95 on this 4-dataset LODO setup.')
print('\nAll numbers leak-guarded (LODO + patient-id assert), all-grades checked, calibration fit on validation only.')

OPTION C FINAL — within-one (single seed -> ensemble -> +calibration):
fold           seed0   ensemble    +calib    exact     qwk
oai            0.825      0.830     0.947    0.303   0.518
nhanes3        0.793      0.759     0.904    0.242   0.515

Best within-1 reached: 0.947
>>> 0.88-0.94 band: strong, but short of 0.95.

All numbers leak-guarded (LODO + patient-id assert), all-grades checked, calibration fit on validation only.
